# 02 — Reproducible Local Modeling Pipeline (Mumbai Air Quality Challenge)

**Architecture:** Thin orchestration notebook importing modular, glass-box functions from `src/`.
- **Hardware Target:** Local CPU (VS Code native per-cell timers displayed automatically beneath each cell).
- **Reproducibility:** Global `SEED = 42`, open-source packages only (no AutoML), zero-overwrite immutable run bundles in `runs/`.
- **Validation Gate:** 6 strictly causal out-of-sample windows (`W1`-`W6`) comparing 4-Branch, 8-Branch (`LGBM + HGB + XGB + CatBoost`), and Aligned 2024 Autumn History modes with Horizon-Regime Gating.

In [ ]:
# Block 0: Local Environment Preflight & Module Imports
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.config import (
    BASE_CAUSAL_CUTOFFS,
    FEATURE_COLS,
    HAS_CAT,
    HAS_XGB,
    PIPELINE_CONFIG,
    PORTFOLIO_PRESETS,
    PROCESSED_DATA_DIR,
    resolve_raw_data_dir,
    run_preflight_check,
)
from src.cleaning import clean_all_datasets, load_raw_core_datasets
from src.features import build_model0_heuristic, build_supervised_dataset, compute_cutoff_priors
from src.cv import fit_multibranch_ensemble, generate_test_predictions, run_6window_cv_tournament
from src.validation import (
    append_experiment_log,
    assert_submission_contract,
    audit_private_block_physics,
    audit_unlocked_public_windows,
    get_git_state,
    save_immutable_run_artifacts,
)

env_info = run_preflight_check()
RAW_DIR = resolve_raw_data_dir()

## 1. Load Frozen Cleaned Data Checkpoints (or Run Cleaning Stage)
Uses frozen `.parquet` files generated by `01_eda_audit.ipynb` if present; otherwise runs `clean_all_datasets()` automatically.

In [ ]:
train_clean_pq = PROCESSED_DATA_DIR / "train_clean.parquet"
hist_2024_pq = PROCESSED_DATA_DIR / "hist_2024_aligned.parquet"

if train_clean_pq.exists():
    train_raw, test_raw, sample_sub_raw, exact_station_map = load_raw_core_datasets(RAW_DIR)
    train_clean = pd.read_parquet(train_clean_pq)
    hist_2024_aligned = pd.read_parquet(hist_2024_pq) if hist_2024_pq.exists() else None
    print(f"[Pipeline Check 1] Loaded frozen Parquet checkpoints | train_clean={train_clean.shape}")
else:
    stages = clean_all_datasets(
        raw_dir=RAW_DIR,
        hist_2024_months=PIPELINE_CONFIG["hist_2024_months"],
        align_method=PIPELINE_CONFIG["align_method_2024"],
        save_parquet=True,
    )
    train_raw = stages["train_raw"]
    train_clean = stages["train_clean"]
    test_raw = stages["test_raw"]
    sample_sub_raw = stages["sample_sub_raw"]
    hist_2024_aligned = stages["hist_2024_aligned"]

# Construct full causal test cutoff schedule ending at the latest validated timestamp
final_cutoff = train_clean["timestamp_ist"].max()
causal_cutoffs = sorted(list(set(BASE_CAUSAL_CUTOFFS + [final_cutoff])))
print(f"[Pipeline Check 1B] Active Final Cutoff: {final_cutoff} ({len(causal_cutoffs)} causal cutoffs configured)")

## 2. Model 0 Zero-ML Heuristic Proof & Glass-Box Feature Inspection
Always verify Model 0 (14-day gap-safe `station x hour` median) first to guarantee zero ID/timestamp bugs, and demonstrate the optional `trace={}` hook for inspecting internal feature tables in VS Code Data Wrangler.

In [ ]:
# Verify Model 0 Baseline against the hard submission contract
sub_model0 = build_model0_heuristic(train_clean, test_raw, causal_cutoffs)
assert_submission_contract(sub_model0, test_raw, sample_sub_raw)
audit_unlocked_public_windows(sub_model0, test_raw, train_raw, label="Model0_14d_Median")

# Glass-box inspection demo: pass trace={} to inspect internal priors/anomalies at final_cutoff
debug_trace = {}
_ = compute_cutoff_priors(train_clean, final_cutoff, trace=debug_trace)
print("\n[Glass-Box Check] Keys captured inside debug_trace (open in VS Code Data Wrangler):", list(debug_trace.keys()))
print(debug_trace["priors_df"].head(6).to_string(index=False))

## 3. Build Gap-Safe Supervised Feature Datasets (Multi-Core CPU)
Generates 64-hour forward supervised windows anchored at `09:00 IST` and `21:00 IST` using $O(\log N)$ calendar slicing (`np.searchsorted`).

In [ ]:
sup_2025_base = build_supervised_dataset(
    df_series=train_clean,
    label="2025_2026_Base",
    is_2024_hist=False,
    clip_cap=PIPELINE_CONFIG["clip_cap"],
    mean_blend=PIPELINE_CONFIG["mean_blend"],
)

sup_2024_hist = (
    build_supervised_dataset(
        df_series=hist_2024_aligned,
        label="2024_Autumn_Aligned",
        is_2024_hist=True,
        clip_cap=PIPELINE_CONFIG["clip_cap"],
        mean_blend=PIPELINE_CONFIG["mean_blend"],
    )
    if hist_2024_aligned is not None
    else pd.DataFrame()
)

if len(sup_2024_hist) > 0:
    assert sup_2024_hist["timestamp_ist"].max() < sup_2025_base["timestamp_ist"].min(), "Temporal overlap leak!"
    print("[Pipeline Check 3] Confirmed strict temporal isolation between 2024 and 2025-2026 supervised sets.")

## 4. 6-Window Out-of-Sample Causal CV Tournament
Compares:
- `M0_Base_2025_26 (4-Branch LGBM+HGB)`
- `M1_Base_2025_26 (8-Branch LGBM+HGB+XGB+CAT)`
- `M3_Hist2024_CeilHr + Base (8-Branch)` (when 2024 archive is present in `data/raw/`)

In [ ]:
candidate_modes = [
    ("M0_Base_2025_26 (4-Branch LGBM+HGB)", sup_2025_base, False),
]
if HAS_XGB or HAS_CAT:
    candidate_modes.append(("M1_Base_2025_26 (8-Branch CPU)", sup_2025_base, True))

if len(sup_2024_hist) > 0:
    sup_comb = pd.concat([sup_2024_hist, sup_2025_base], ignore_index=True)
    sup_comb["location_id"] = sup_comb["location_id"].astype("category")
    candidate_modes.append(("M3_Hist2024_CeilHr + Base (8-Branch)", sup_comb, HAS_XGB or HAS_CAT))

cv_summary = run_6window_cv_tournament(
    candidate_modes=candidate_modes,
    train_clean=train_clean,
    hist_2024_weight=PIPELINE_CONFIG["hist_2024_sample_weight"],
    benchmark_rmse=PIPELINE_CONFIG["benchmark_cv_rmse"],
)
winner = cv_summary["winner"]

## 5. Full Refit on Winning Configuration & Feature Importance Audit
Refits the winning multi-branch architecture on the complete causal supervised set and audits top split importances across Additive Huber and Multiplicative Log-Ratio models.

In [ ]:
win_sup_df = winner["sup_df"]
win_sw = None
if "is_2024_hist" in win_sup_df.columns and (win_sup_df["is_2024_hist"] == 1).any():
    win_sw = np.where(win_sup_df["is_2024_hist"].to_numpy() == 1, PIPELINE_CONFIG["hist_2024_sample_weight"], 1.0)

full_models = fit_multibranch_ensemble(
    tr_df=win_sup_df,
    sample_weight=win_sw,
    include_xgb_cat=winner["include_xgb_cat"],
)
print(f"[Pipeline Check 5A] Refitted Full Ensemble | Duan Smearing (LGB) = {full_models['smear_lgb']:.4f}")

imp_df = pd.DataFrame(
    {
        "LGBM_Huber_Splits": full_models["m_huber"].feature_importances_,
        "LGBM_LogRatio_Splits": full_models["m_log_lgb"].feature_importances_,
    },
    index=FEATURE_COLS,
).sort_values("LGBM_Huber_Splits", ascending=False)
print("\n[Pipeline Check 5B] Top 12 Features in Refitted Full Models:")
print(imp_df.head(12).to_string())

## 6. Two-Slot Portfolio Inference, Hard Validation Checks & Immutable Run Export
Generates both the **CV-Optimal Gated Hybrid (Slot 1)** and the **Winter Inversion Hedge (Slot 2)**, validates format contracts and diurnal physics, writes immutable zero-overwrite bundles into `runs/`, and appends a record to `docs/02_Experiment_Log.csv`.

In [ ]:
# Slot 1: CV-Optimal Balanced Sea-Breeze Dispersion Gated Hybrid
s1_cfg = PORTFOLIO_PRESETS["slot1_balanced"]
slot1_detail = generate_test_predictions(
    full_models=full_models,
    train_clean=train_clean,
    test_raw=test_raw,
    causal_cutoffs=causal_cutoffs,
    clip_cap=s1_cfg["clip_cap"],
    mean_blend=s1_cfg["mean_blend"],
    prior_weights=s1_cfg["prior_weights"],
    tau_fast=s1_cfg["tau_fast"],
    tau_slow=s1_cfg["tau_slow"],
    scheme_name=winner["best_scheme_name"],
    lw_param=winner["best_lw_param"],
    gated_params=winner["gated_params"],
    res_scale=s1_cfg["res_scale"],
)
slot1_sub = pd.DataFrame({"ID": test_raw["ID"], "Target": slot1_detail["pred_target"]})

# Hard assertions + Physics Audit + Unlocked Window Audit
assert_submission_contract(slot1_sub, test_raw, sample_sub_raw)
audit_private_block_physics(slot1_detail, label="Slot1_Gated_Hybrid")
unlocked_aud_s1 = audit_unlocked_public_windows(slot1_sub, test_raw, train_raw, label="Slot1_Gated_Hybrid")

# Save zero-overwrite immutable bundle and auto-log to docs/02_Experiment_Log.csv
metrics_payload = {
    "winning_mode": winner["mode_name"],
    "blend_scheme": winner["best_scheme_name"],
    "gated_params": winner["gated_params"],
    "pooled_cv_rmse": round(winner["best_rmse"], 4),
    "w1_w5_cv_rmse": round(winner["rmse_w1_w5"], 4),
    "runtime_sec": winner["runtime_sec"],
    "unlocked_windows": unlocked_aud_s1,
}
sub1_path = save_immutable_run_artifacts(
    run_label="slot1_gated_hybrid",
    sub_df=slot1_sub,
    metrics_dict=metrics_payload,
    config_dict={**PIPELINE_CONFIG, **s1_cfg},
    oof_df=winner["oof_all"],
)

git_info = get_git_state()
append_experiment_log(
    {
        "run_timestamp": pd.Timestamp.now(tz="Asia/Kolkata").strftime("%Y-%m-%d %H:%M:%S"),
        "mode_name": winner["mode_name"],
        "blend_scheme": winner["best_scheme_name"],
        "pooled_6win_cv_rmse": round(winner["best_rmse"], 4),
        "w1_w5_cv_rmse": round(winner["rmse_w1_w5"], 4),
        "w5_nov08_10_raw_rmse": unlocked_aud_s1["W5_Nov08_10_LB"]["raw_rmse"],
        "runtime_sec": winner["runtime_sec"],
        "git_branch": git_info["branch"],
        "git_commit": git_info["commit"],
        "submission_file": sub1_path.name,
    }
)

In [ ]:
# Optional Slot 2: Persistent Winter Inversion Hedge (Structurally Hedged Portfolio Pair)
s2_cfg = PORTFOLIO_PRESETS["slot2_inversion_hedge"]
slot2_detail = generate_test_predictions(
    full_models=full_models,
    train_clean=train_clean,
    test_raw=test_raw,
    causal_cutoffs=causal_cutoffs,
    clip_cap=s2_cfg["clip_cap"],
    mean_blend=s2_cfg["mean_blend"],
    prior_weights=s2_cfg["prior_weights"],
    tau_fast=s2_cfg["tau_fast"],
    tau_slow=s2_cfg["tau_slow"],
    scheme_name=winner["best_scheme_name"],
    lw_param=winner["best_lw_param"],
    gated_params=winner["gated_params"],
    res_scale=s2_cfg["res_scale"],
)
slot2_sub = pd.DataFrame({"ID": test_raw["ID"], "Target": slot2_detail["pred_target"]})

assert_submission_contract(slot2_sub, test_raw, sample_sub_raw)
audit_private_block_physics(slot2_detail, label="Slot2_Inversion_Hedge")
unlocked_aud_s2 = audit_unlocked_public_windows(slot2_sub, test_raw, train_raw, label="Slot2_Inversion_Hedge")

save_immutable_run_artifacts(
    run_label="slot2_inversion_hedge",
    sub_df=slot2_sub,
    metrics_dict={**metrics_payload, "unlocked_windows": unlocked_aud_s2},
    config_dict={**PIPELINE_CONFIG, **s2_cfg},
)